# TV Denoising on the Hyperbolic Manifold

This notebook runs the total variation (TV) denoising experiment on the hyperbolic manifold $\mathbb{H}^2$.
The objective combines an $L^2$ fidelity term with a total variation regularizer:
$$f(p) = \frac{1}{n}\left(\frac{1}{2}\, d_{\mathcal{M}}(p, q)^2 + \alpha\, \mathrm{TV}(p)\right)$$
where $q$ is the noisy signal and $p$ lives on the power manifold $(\mathbb{H}^2)^n$.

**Methods compared:** RPB, SGM (and optionally RCBM, PBA).

> **Note:** The full experiment (100,000+ iterations, all methods, CSV export) is run via `julia experiment_denoising_hyperbolic.jl` from the repository root. This notebook runs a shorter version for interactive exploration.

In [ ]:
using LinearAlgebra, Random, Statistics
using ManifoldDiff, Manifolds, ManifoldsBase, Manopt, ManoptExamples
using LRUCache
using Plots

include("../src/RiemannianProximalBundle.jl")

# Inner product helper required by the RPB solver
inner_product(M, p, X, Y) = inner(M, p, X, Y)

## Parameters

Modify the values below to configure the experiment.

In [ ]:
# ── Signal parameters ────────────────────────────────────────────────────────
num_points = 64      # Signal discretization (use 496 for paper results; smaller = faster)
sigma      = 0.3     # Noise level (σ of Riemannian Gaussian)
alpha      = 0.5     # TV regularization strength

# ── Solver parameters ────────────────────────────────────────────────────────
atol               = 1e-8      # Convergence tolerance
max_iter           = 5_000     # Max iterations (use 100_000 for paper results)
back_tracking_fact = 2         # Backtracking factor for proximal parameter

# ── RPB-specific parameters ──────────────────────────────────────────────────
proximal_parameter = 1.0 / num_points   # Initial proximal parameter ρ₀
trust_parameter    = 0.001              # Descent condition threshold β
retraction_error   = 1.0               # Retraction error constant
transport_error    = 1.0               # Transport error constant
sectional_curvature = -1.0             # Sectional curvature of ℍ²

Random.seed!(42)
println("Configuration: num_points=$num_points, sigma=$sigma, alpha=$alpha, max_iter=$max_iter")

## Signal Generation and Manifold Setup

In [ ]:
# Generate a piecewise-geodesic signal on ℍ² with jump discontinuities
function artificial_H2_signal(pts::Integer=100, a::Real=0.0, b::Real=1.0, T::Real=(b-a)/2)
    t = range(a, b; length=pts)
    x = [[s, sign((2*pi/T)*s)] for s in t]
    y = vcat([x[1]],
             [x[i] for i in 2:(length(x)-1) if (x[i][2] != x[i+1][2] || x[i][2] != x[i-1][2])],
             [x[end]])
    y = map(z -> Manifolds._hyperbolize(Hyperbolic(2), z), y)
    l = Int(round(pts * T / (2*(b-a))))
    data = []
    for i in 1:2:(length(y)-1)
        append!(data, shortest_geodesic(Hyperbolic(2), y[i], y[i+1], range(0.0, 1.0; length=l)))
    end
    return data
end

H      = Hyperbolic(2)
signal = artificial_H2_signal(num_points, -6.0, 6.0, 3)
noise  = map(p -> exp(H, p, rand(H; vector_at=p, σ=sigma)), signal)
Hn     = PowerManifold(H, NestedPowerRepresentation(), length(noise))

println("Signal length: $(length(signal))")

# Polish parallel transport with projection for numerical stability
struct ProjectedParallelTransport <: AbstractVectorTransportMethod end
function ManifoldsBase.vector_transport_to!(M::Hyperbolic, Y, p, X, q, ::ProjectedParallelTransport)
    parallel_transport_to!(M, Y, p, X, q)
    project!(M, Y, q, Y)
    return Y
end

# Polish the exponential retraction to enforce hyperboloid normalization
function ManifoldsBase.retract!(M::Hyperbolic, q, p, X, ::ExponentialRetraction)
    exp!(M, q, p, X)
    mq = dot(@view(q[1:end-1]), @view(q[1:end-1])) - q[end]^2
    q ./= sqrt(-mq)
    return q
end

# Objective and subgradient
function f_obj(M, p)
    return 1/length(noise) * (
        0.5 * distance(M, noise, p)^2 +
        alpha * ManoptExamples.Total_Variation(M, p)
    )
end

function subgradient_of_f(M, p)
    g = 1/length(noise) * (
        ManifoldDiff.grad_distance(M, noise, p) +
        alpha * ManoptExamples.subgrad_Total_Variation(M, p; atol=atol)
    )
    return project(M, p, g)
end

# Retraction and transport maps for the power manifold
retraction_map       = (p, v) -> retract(Hn, p, v, ExponentialRetraction())
transport_map        = (p, q, v) -> vector_transport_to(Hn, p, v, q, ProjectionTransport())

initial_objective   = f_obj(Hn, noise)
initial_subgradient = subgradient_of_f(Hn, noise)
println("Initial objective: $initial_objective")

## Run Solvers

In [ ]:
# ── RPB ───────────────────────────────────────────────────────────────────────
println("Running RPB...")
rpb_start = time()
rpb_solver = RProximalBundle(
    Hn, retraction_map, transport_map,
    (x) -> f_obj(Hn, x), (x) -> subgradient_of_f(Hn, x),
    noise, initial_objective, initial_subgradient;
    max_iter=max_iter, tolerance=atol,
    proximal_parameter=proximal_parameter, trust_parameter=trust_parameter,
    retraction_error=retraction_error, transport_error=transport_error,
    sectional_curvature=sectional_curvature,
    back_tracking_factor=Float64(back_tracking_fact),
    know_minimizer=false, relative_error=false,
    memory_lite=true
)
run!(rpb_solver)
rpb_time = time() - rpb_start
println("  RPB: $(length(rpb_solver.raw_objective_history)-1) iterations in $(round(rpb_time, digits=2))s")
println("  Final objective: $(rpb_solver.raw_objective_history[end])")

# ── SGM (Subgradient Method) ──────────────────────────────────────────────────
println("Running SGM...")
sgm_start = time()
sgm = subgradient_method(Hn, f_obj, subgradient_of_f, noise;
    cache=(:LRU, [:Cost, :SubGradient], 50),
    stepsize=DecreasingLength(; exponent=1, factor=1, subtrahend=0, length=1, shift=0, type=:absolute),
    record=[:Iteration, :Cost], return_state=true,
    stopping_criterion=StopAfterIteration(max_iter)
)
sgm_time = time() - sgm_start
sgm_record = get_record(sgm)
println("  SGM: $(length(sgm_record)) iterations in $(round(sgm_time, digits=2))s")

println("All solvers finished.")

## Results

In [ ]:
# Plot objective vs. iteration
p_plot = plot(title="TV Denoising on ℍ² ($num_points points, σ=$sigma, α=$alpha)",
              xlabel="Iteration", ylabel="Objective f(p)",
              yscale=:log10, legend=:topright, dpi=150)

rpb_iters = 0:length(rpb_solver.raw_objective_history)-1
sgm_iters = [r[1] for r in sgm_record]
sgm_objs  = [r[2] for r in sgm_record]

plot!(p_plot, rpb_iters, rpb_solver.raw_objective_history, label="RPB", color=:blue, linewidth=2)
plot!(p_plot, sgm_iters, sgm_objs,                         label="SGM", color=:orange, linewidth=1)

display(p_plot)

In [ ]:
# Wall-clock time plot
p_time = plot(title="TV Denoising on ℍ² — Wall-clock time",
              xlabel="Time (s)", ylabel="Objective f(p)",
              yscale=:log10, legend=:topright, dpi=150)

# Approximate uniform time for RPB (time / iters)
rpb_n = length(rpb_solver.raw_objective_history)
rpb_times = range(0, rpb_time; length=rpb_n)
sgm_n = length(sgm_objs)
sgm_times = range(0, sgm_time; length=sgm_n)

plot!(p_time, rpb_times, rpb_solver.raw_objective_history, label="RPB", color=:blue, linewidth=2)
plot!(p_time, sgm_times, sgm_objs,                         label="SGM", color=:orange, linewidth=1)

display(p_time)